# TerraWatch Milestone 4 — Full-Data RSCaMa Paired Change Captioner

This notebook reproduces the full-data LEVIR-MCI change-captioning experiments (**6,444 train pairs / 32,220 train captions, 1,242 val pairs, 1,858 held-out test pairs**) with:
- Strict **train-only vocabulary** (`388` words, `min_word_freq=5`, built exclusively on the 6,444 training pairs — zero val/test leakage)
- Exact **Triton fused selective-scan kernel** (`_TritonSelectiveScan` in `scripts/train_captioner_full.py`) delivering **10× faster** forward+backward passes than the Python reference loop (`< 5e-7` numerical agreement) without requiring `mamba_ssm` wheel compilation
- Controlled ablations `C01`–`C07`, released benchmark `C00_rscama_released`, 3-seed validation on winner **`C06_rscama_frozen_clip`**, and held-out Test evaluation (1,858 pairs)

By default (`RUN_TRAINING = False`), the notebook replays all committed receipts from `docs/milestone4/evidence/full_data_m4/`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content/TerraWatch")]
REPO = next((p for p in CANDIDATES if (p / "scripts/train_captioner_full.py").is_file()), None)
if REPO is None:
    REPO = Path("/content/TerraWatch")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/shoaib2000857/Group-7-DS-and-AI-Lab-Project.git", str(REPO)],
        check=True,
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

EVIDENCE_DIR = REPO / "docs/milestone4/evidence/full_data_m4"
CFG = json.loads((REPO / "configs/captioner_training.json").read_text(encoding="utf-8"))
print("Selected deployment captioner:", CFG["selected_run_id"], "| SHA-256:", CFG["selected_checkpoint_sha256"][:16])

## 1. Replay Controlled RSCaMa Experiments, 3-Seed Stability & Held-Out Test

In [ ]:
agg = json.loads((EVIDENCE_DIR / "aggregated_summary.json").read_text(encoding="utf-8"))
print(f"{'Run ID':<28} {'LR':<7} {'Opt':<6} {'Freeze':<7} {'Ep':>3} {'Val B4':>8} {'ROUGE-L':>8} {'Chg B4':>8} {'NC FA':>7}")
print("-" * 92)
for r in agg["captioner_ranking"]:
    print(
        f"{r['run_id']:<28} {r['lr']:<7} {r['optimizer']:<6} {str(r['freeze_encoder']):<7} "
        f"{r['best_epoch']:>3} {r['val_bleu4']:>8.4f} {r['val_rouge_l']:>8.4f} "
        f"{r['val_changed_bleu4']:>8.4f} {r['nc_fa_rate_pct']:>6.2f}%"
    )

print("\n3-Seed Stability (C06_rscama_frozen_clip, seeds 42, 0, 2026):")
for name, st in agg["captioner_multiseed"].items():
    print(f"  {name}: Val BLEU-4 = {st['val_bleu4_mean']:.4f} ± {st['val_bleu4_std']:.4f} | Val ROUGE-L = {st['val_rouge_l_mean']:.4f} ± {st['val_rouge_l_std']:.4f}")

test_eval = json.loads((EVIDENCE_DIR / "captioner_runs/C06_rscama_frozen_clip/eval_test.json").read_text(encoding="utf-8"))
rel_test = agg.get("released_rscama_test") or {}
print("\nHeld-Out LEVIR-MCI Test (1,858 pairs):")
print(f"  Selected C06_rscama_frozen_clip: BLEU-4={test_eval['bleu4']:.4f}, ROUGE-L={test_eval['rouge_l']:.4f}, NC FA={test_eval['unchanged_fa_rate']*100:.2f}%, Missed={test_eval['changed_miss_rate']*100:.2f}%")
if rel_test:
    print(f"  Released RSCaMa (Benchmark):     BLEU-4={rel_test['bleu4']:.4f}, ROUGE-L={rel_test['rouge_l']:.4f}, NC FA={rel_test['unchanged_fa_rate']*100:.2f}%, Missed={rel_test['changed_miss_rate']*100:.2f}%")

## 2. Train Selected Captioner (`C06_rscama_frozen_clip`) & Evaluate Held-Out Test on GPU

Requires a CUDA GPU (T4 or L4) and the extracted LEVIR-MCI dataset plus the upstream `RSCaMa` repo checkout (`git clone https://github.com/Chen-Yang-Liu/RSCaMa.git`).

In [ ]:
RUN_TRAINING = False
DATA_DIR = Path(os.environ.get("TERRAWATCH_MCI_DIR", "data/processed/levir_mci"))
RSCAMA_SRC = Path(os.environ.get("RSCAMA_DIR", "/content/RSCaMa"))
OUT_DIR = Path("outputs/captioner_runs/C06_rscama_frozen_clip")

train_cmd = [
    sys.executable, "scripts/train_captioner_full.py",
    "--run-id", "C06_rscama_frozen_clip",
    "--data-dir", str(DATA_DIR),
    "--rscama-source-dir", str(RSCAMA_SRC),
    "--out-dir", str(OUT_DIR),
    "--epochs", "15",
    "--batch-size", "32",
    "--lr", "1e-4",
    "--optimizer", "adamw",
    "--weight-decay", "1e-2",
    "--scheduler", "cosine",
    "--label-smoothing", "0.0",
    "--freeze-encoder",
    "--seed", "42",
    "--resume",
]
test_cmd = [
    sys.executable, "scripts/train_captioner_full.py",
    "--eval-only",
    "--eval-split", "test",
    "--checkpoint", str(OUT_DIR / "best.pth"),
    "--data-dir", str(DATA_DIR),
    "--rscama-source-dir", str(RSCAMA_SRC),
    "--out-dir", str(OUT_DIR),
]
print("Train command:\n ", " ".join(train_cmd))
print("\nHeld-out Test command:\n ", " ".join(test_cmd))
if RUN_TRAINING:
    subprocess.run(train_cmd, check=True)
    subprocess.run(test_cmd, check=True)